### Implement Your Own Retriever

In [3]:
# Install only if needed
# %pip install transformers torch --quiet

import re
from transformers import pipeline

In [4]:
docs = [
    "AI helps doctors detect diseases from medical images.",
    "Machine learning can predict patient health risks.",
    "Computer vision is used to analyze X-rays and MRI scans.",
    "NLP can extract useful information from medical reports."
]

query = "How is AI used in healthcare?"

In [5]:
def tokenize(text):
    text = text.lower()
    words = re.findall(r"\b\w+\b", text)
    return words


def retrieve_context(query, docs, k=1):
    query_words = set(tokenize(query))

    scores = []

    for doc in docs:
        doc_words = set(tokenize(doc))

        score = 0
        for word in query_words:
            if word in doc_words:
                score += 1

        scores.append((score, doc))

    scores = sorted(scores, key=lambda x: x[0], reverse=True)

    top_docs = []
    for score, doc in scores[:k]:
        top_docs.append(doc)

    return top_docs


context = retrieve_context(query, docs)
print("Retrieved context:", context)

Retrieved context: ['Computer vision is used to analyze X-rays and MRI scans.']


In [6]:
# docs = [
#     "RAG stands for Retrieval-Augmented Generation.",
#     "It combines information retrieval and text generation.",
#     "LangChain can be used to build RAG pipelines easily.",
#     "Transformers use self-attention to handle sequential data."
# ]

# query = "What is RAG?"

# # TODO: Implement a function that returns the most relevant doc(s)
# def retrieve_context(query, docs, k=1):
#     query_words = query.lower().replace("?", "").split()
#     scores = []

#     for doc in docs:
#         score = 0
#         for word in query_words:
#             if word in doc.lower():
#                 score += 1

#         scores.append((score, doc))

#     scores.sort(reverse=True)
#     return [doc for score, doc in scores[:k]]
#     # HINT: break query and docs into lowercase words
#     # Count how many query words appear in each doc
#     # Sort by score and return top-k docs
#     # pass

# context = retrieve_context(query, docs)
# print("Retrieved context:", context)






docs = [
    "RAG stands for Retrieval-Augmented Generation.",
    "It combines information retrieval and text generation.",
    "LangChain can be used to build RAG pipelines easily.",
    "Transformers use self-attention to handle sequential data."
]

query = "What is RAG?"

# TODO: Implement a function that returns the most relevant doc(s)
def retrieve_context(query, docs, k=1):
    # HINT: break query and docs into lowercase words
    # Count how many query words appear in each doc
    # Sort by score and return top-k docs
    import re

docs = [
    "RAG stands for Retrieval-Augmented Generation.",
    "It combines information retrieval and text generation.",
    "LangChain can be used to build RAG pipelines easily.",
    "Transformers use self-attention to handle sequential data."
]

query = "What is RAG?"

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def retrieve_context(query, docs, k=1):
    query_words = set(tokenize(query))
    scores = []

    for doc in docs:
        doc_words = set(tokenize(doc))
        score = len(query_words.intersection(doc_words))
        scores.append((score, doc))

    scores = sorted(scores, key=lambda x: x[0], reverse=True)
    top_docs = [doc for score, doc in scores[:k]]

    return top_docs

context = retrieve_context(query, docs)
print("Retrieved context:", context)

context = retrieve_context(query, docs)
print("Retrieved context:", context)

Retrieved context: ['RAG stands for Retrieval-Augmented Generation.']
Retrieved context: ['RAG stands for Retrieval-Augmented Generation.']


### Write the Prompt Template

In [7]:
def make_prompt(context, question):
    context_text = "\n".join(context)

    prompt = f"""
Use the given context to answer the question.

Context:
{context_text}

Question:
{question}

Answer:
"""
    return prompt


prompt_text = make_prompt(context, query)
print(prompt_text)


Use the given context to answer the question.

Context:
RAG stands for Retrieval-Augmented Generation.

Question:
What is RAG?

Answer:



## Plug in a HuggingFace Model
Experiment with temperature (0.2, 0.8) and observe changes.

In [8]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
hf_gen = pipeline(
    task="text-generation",
    model="distilgpt2",
    max_new_tokens=50
)

response = hf_gen(
    prompt_text,
    pad_token_id=hf_gen.tokenizer.eos_token_id
)[0]["generated_text"]

print(response)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6431.55it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is des


Use the given context to answer the question.

Context:
RAG stands for Retrieval-Augmented Generation.

Question:
What is RAG?

Answer:
RAG is a reference to the ability to add data to a global object.
RAG is a reference to the ability to add data to a global object.
RAG is a reference to the ability to add data to a global object.


In [10]:
response_temp_02 = hf_gen(
    prompt_text,
    max_new_tokens=50,
    temperature=0.2,
    do_sample=True,
    pad_token_id=hf_gen.tokenizer.eos_token_id
)[0]["generated_text"]

print("Temperature 0.2 Output:")
print(response_temp_02)

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temperature 0.2 Output:

Use the given context to answer the question.

Context:
RAG stands for Retrieval-Augmented Generation.

Question:
What is RAG?

Answer:
RAG is a new generation of RAG.
Question:
What is RAG?
Answer:
RAG is a new generation of RAG.
Question:
What is RAG?
Answer:
RAG is a


In [11]:
response_temp_08 = hf_gen(
    prompt_text,
    max_new_tokens=50,
    temperature=0.8,
    do_sample=True,
    pad_token_id=hf_gen.tokenizer.eos_token_id
)[0]["generated_text"]

print("Temperature 0.8 Output:")
print(response_temp_08)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temperature 0.8 Output:

Use the given context to answer the question.

Context:
RAG stands for Retrieval-Augmented Generation.

Question:
What is RAG?

Answer:
That is the case for the RAG system, because ARS is a fully interoperable, simple, and non-linear digital format.
What is ARS, for example, like?
Answer:
ARS does not have a single


In [12]:
print("""
Observation:
At temperature 0.2, the answer is usually more controlled and less random.
At temperature 0.8, the answer becomes more creative, but it can also become less stable.
""")


Observation:
At temperature 0.2, the answer is usually more controlled and less random.
At temperature 0.8, the answer becomes more creative, but it can also become less stable.



## Connect It All: Build a Mini RAG Function

In [13]:
def mini_rag(query):
    context = retrieve_context(query, docs)
    prompt = make_prompt(context, query)

    result = hf_gen(
        prompt,
        max_new_tokens=50,
        pad_token_id=hf_gen.tokenizer.eos_token_id
    )[0]["generated_text"]

    return result


print(mini_rag("How do transformers work?"))

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the given context to answer the question.

Context:
Transformers use self-attention to handle sequential data.

Question:
How do transformers work?

Answer:
Transformers use self-attention to handle sequential data.
Question:
How do transformers work?
Answer:
Transformers use self-attention to handle sequential data.
Question:
How do these transformers work?

